# CE49X Final Project

## Conflict Situation Monitoring for Maritime Shipping

This notebook is the complete output-oriented submission notebook. It contains the code needed to load the final project outputs, the required analysis tables, all generated figures, and the written discussion sections. The full end-to-end data collection and processing pipeline is provided separately in `CE49X_Final_Project_Run_All.py`.

**Completed dataset summary**

| table | rows |
| --- | --- |
| firms_detections | 265321 |
| news_articles | 3724 |
| thermal_events | 2904 |
| event_matches | 135311 |

**Headline result:** NASA FIRMS thermal anomalies can support conflict-risk monitoring when combined with news matching. The strongest conflict-news association was found in **Ukraine_Black_Sea** (98.7%), and the best classifier was **SVM** with F1 = **0.950**.


## Region Selection and Rationale

The selected regions connect armed-conflict risk with maritime shipping, energy supply, route disruption, insurance premiums, and fuel-price volatility.

| Region | Why It Matters |
| --- | --- |
| Ukraine / Black Sea | Black Sea grain and energy-linked trade, port disruption risk, and Russia-Ukraine war activity. |
| Red Sea / Yemen | Bab el-Mandeb and Suez-linked route risk, rerouting costs, and attacks affecting vessel insurance. |
| Persian Gulf / Hormuz | Major oil export corridor where conflict or facility fires can influence global energy prices. |
| Eastern Mediterranean | Conflict exposure near port systems, offshore energy interests, and Suez-adjacent shipping flows. |


## Reproducibility Code

Run the full pipeline from a terminal with:

```bash
cd /Users/yigitbeyzadeoglu/Documents/Codex/2026-06-01/files-mentioned-by-the-user-final/outputs
source .venv/bin/activate
docker compose up -d
python -u CE49X_Final_Project_Run_All.py
```

The script downloads FIRMS data, collects news, writes PostgreSQL tables, clusters thermal events, matches events to news, trains ML models, and saves `dashboard.png`.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
DATA = ROOT / "final_data_snapshots"

firms = pd.read_csv(DATA / "firms_detections.csv")
news = pd.read_csv(DATA / "news_articles.csv")
events = pd.read_csv(DATA / "thermal_events.csv")
matches = pd.read_csv(DATA / "event_matches.csv")
table_counts = pd.read_csv(DATA / "table_counts.csv")
coverage = pd.read_csv(DATA / "region_coverage_summary.csv")
monthly = pd.read_csv(DATA / "monthly_summary.csv")
hypothesis = pd.read_csv(DATA / "hypothesis_test.csv")
ml_results = pd.read_csv(DATA / "ml_model_results.csv")
source_counts = pd.read_csv(DATA / "news_source_counts.csv")

table_counts


## Task 1: Data Collection and Assembly

### Sources Used

- **NASA FIRMS area API**: VIIRS_SNPP_SP active fire / thermal anomaly detections.
- **GDELT DOC 2.1 API**: conflict-related news articles.
- **Google News RSS**: conflict-related news search results.
- **Bing News RSS**: additional conflict-related news source.

Date range: **2024-01-01 to 2024-06-30**.

### FIRMS Data Sample

| latitude | longitude | bright_ti4 | scan | track | acq_date | acq_time | satellite | instrument | confidence | version | bright_ti5 | frp | daynight | type | region | firms_source | api_url | brightness | acq_datetime |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 52.536 | 39.632 | 328.020 | 0.450 | 0.630 | 2024-01-01 | 45 | N | VIIRS | n | 2 | 271.100 | 4.070 | N | 2 | Ukraine_Black_Sea | VIIRS_SNPP_SP | https://firms.modaps.eosdis.nasa.gov/api/area/csv/[MAP_KEY]/VIIRS_SNPP_SP/22.0,44.0,41.0,53.0/5/2024-01-01 | 328.020 | 2024-01-01 00:45:00+00:00 |
| 52.536 | 39.631 | 330.390 | 0.450 | 0.630 | 2024-01-01 | 45 | N | VIIRS | n | 2 | 271.340 | 2.810 | N | 2 | Ukraine_Black_Sea | VIIRS_SNPP_SP | https://firms.modaps.eosdis.nasa.gov/api/area/csv/[MAP_KEY]/VIIRS_SNPP_SP/22.0,44.0,41.0,53.0/5/2024-01-01 | 330.390 | 2024-01-01 00:45:00+00:00 |
| 52.544 | 39.646 | 301.100 | 0.450 | 0.630 | 2024-01-01 | 45 | N | VIIRS | n | 2 | 268.840 | 1.000 | N | 2 | Ukraine_Black_Sea | VIIRS_SNPP_SP | https://firms.modaps.eosdis.nasa.gov/api/area/csv/[MAP_KEY]/VIIRS_SNPP_SP/22.0,44.0,41.0,53.0/5/2024-01-01 | 301.100 | 2024-01-01 00:45:00+00:00 |
| 52.544 | 39.648 | 298.270 | 0.450 | 0.630 | 2024-01-01 | 45 | N | VIIRS | n | 2 | 268.700 | 0.750 | N | 0 | Ukraine_Black_Sea | VIIRS_SNPP_SP | https://firms.modaps.eosdis.nasa.gov/api/area/csv/[MAP_KEY]/VIIRS_SNPP_SP/22.0,44.0,41.0,53.0/5/2024-01-01 | 298.270 | 2024-01-01 00:45:00+00:00 |
| 52.546 | 39.642 | 301.950 | 0.450 | 0.630 | 2024-01-01 | 45 | N | VIIRS | n | 2 | 269.200 | 1.060 | N | 2 | Ukraine_Black_Sea | VIIRS_SNPP_SP | https://firms.modaps.eosdis.nasa.gov/api/area/csv/[MAP_KEY]/VIIRS_SNPP_SP/22.0,44.0,41.0,53.0/5/2024-01-01 | 301.950 | 2024-01-01 00:45:00+00:00 |
| 52.546 | 39.640 | 303.560 | 0.450 | 0.630 | 2024-01-01 | 45 | N | VIIRS | n | 2 | 269.130 | 1.000 | N | 2 | Ukraine_Black_Sea | VIIRS_SNPP_SP | https://firms.modaps.eosdis.nasa.gov/api/area/csv/[MAP_KEY]/VIIRS_SNPP_SP/22.0,44.0,41.0,53.0/5/2024-01-01 | 303.560 | 2024-01-01 00:45:00+00:00 |
| 52.548 | 39.636 | 297.780 | 0.450 | 0.630 | 2024-01-01 | 45 | N | VIIRS | n | 2 | 267.650 | 1.060 | N | 2 | Ukraine_Black_Sea | VIIRS_SNPP_SP | https://firms.modaps.eosdis.nasa.gov/api/area/csv/[MAP_KEY]/VIIRS_SNPP_SP/22.0,44.0,41.0,53.0/5/2024-01-01 | 297.780 | 2024-01-01 00:45:00+00:00 |
| 52.555 | 39.617 | 307.970 | 0.450 | 0.630 | 2024-01-01 | 45 | N | VIIRS | n | 2 | 269.930 | 1.370 | N | 2 | Ukraine_Black_Sea | VIIRS_SNPP_SP | https://firms.modaps.eosdis.nasa.gov/api/area/csv/[MAP_KEY]/VIIRS_SNPP_SP/22.0,44.0,41.0,53.0/5/2024-01-01 | 307.970 | 2024-01-01 00:45:00+00:00 |

### News Data Sample

| title | published_date | source | url | region | location_mentions | snippet | collection_source | access_date |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Российские удары по Киеву сравнили с Армагеддоном , а наш боец в одиночку атаковал позиции боевиков ВСУ : обновленная карта боевых действий спецоперации на Украине 6 . 01 . 24 , главные события дня и фронтовые сводки 6 января в мире | 2024-01-06 06:00:00+00:00 | pronedra.ru | https://pronedra.ru/rossijskij-boecz-v-odinochku-atakoval-poziczii-vsu-obnovlennaya-karta-voennoj-operaczii-na-ukraine-i-sobytij-v-mire-6-yanvarya-2024-goda-714271.html | Ukraine_Black_Sea |  | 20240106T060000Z | GDELT DOC 2.1 API | 2026-06-01 |
| Ворог намагався прорвати оборону ЗСУ на чотирьох напрямках – Генштаб | 2024-02-01 18:45:00+00:00 | glavcom.ua | https://glavcom.ua/country/incidents/voroh-namahavsja-prorvati-oboronu-zsu-na-chotirokh-naprjamkakh-henshtab-983743.html | Ukraine_Black_Sea |  | 20240201T184500Z | GDELT DOC 2.1 API | 2026-06-01 |
| Ситуація на фронті 1 лютого - актуальне зведення Генштабу | 2024-02-01 19:00:00+00:00 | rbc.ua | https://www.rbc.ua/rus/news/situatsiya-fronti-sogodni-1-lyutogo-1706809783.html | Ukraine_Black_Sea |  | 20240201T190000Z | GDELT DOC 2.1 API | 2026-06-01 |
| Генштаб : війська РФ вночі атакували Україну 35 ударними дронами | 2024-01-30 09:45:00+00:00 | radiosvoboda.org | https://www.radiosvoboda.org/a/news-henshtab-shahedy/32797492.html | Ukraine_Black_Sea |  | 20240130T094500Z | GDELT DOC 2.1 API | 2026-06-01 |
| За добу ЗСУ знищили близько тисячі окупантів , 30 артсистем і винищувач Су - 34 | 2024-01-30 09:45:00+00:00 | volynnews.com | https://www.volynnews.com/news/all/za-dobu-zsu-znyshchyly-blyzko-tysiachi-okupantiv-30-artsystem-i-vynyshchuvach-su-34/ | Ukraine_Black_Sea |  | 20240130T094500Z | GDELT DOC 2.1 API | 2026-06-01 |
| Ситуація на фронті - окупанти завдали 81 авіаудар за день - зведення Генштабу за 23 січня | 2024-01-23 19:15:00+00:00 | war.obozrevatel.com | https://war.obozrevatel.com/ukr/okupanti-zavdali-81-aviaudar-za-den-sili-oboroni-znischili-zasobi-ppo-j-artsistemi-voroga-genshtab.htm | Ukraine_Black_Sea |  | 20240123T191500Z | GDELT DOC 2.1 API | 2026-06-01 |
| BlackSeaNews | Оперативна інформація станом на 18 . 00 01 . 02 . 2024 щодо російського вторгнення | 2024-02-01 19:30:00+00:00 | blackseanews.net | https://www.blackseanews.net/read/213708 | Ukraine_Black_Sea |  | 20240201T193000Z | GDELT DOC 2.1 API | 2026-06-01 |
| Втрати Росії в Україні - ЗСУ знищили майже 400 окупантів та 42 одиниці техніки на Таврійському напрямку за добу | 2024-01-10 09:00:00+00:00 | rbc.ua | https://www.rbc.ua/rus/news/znishchena-tehnika-ta-mayzhe-400-rosiyan-1704870392.html | Ukraine_Black_Sea |  | 20240110T090000Z | GDELT DOC 2.1 API | 2026-06-01 |

### News Source Counts

| region | collection_source | article_count |
| --- | --- | --- |
| Eastern_Mediterranean | GDELT DOC 2.1 API | 396 |
| Eastern_Mediterranean | Google News RSS | 86 |
| Persian_Gulf | GDELT DOC 2.1 API | 968 |
| Persian_Gulf | Google News RSS | 100 |
| Red_Sea_Yemen | GDELT DOC 2.1 API | 500 |
| Red_Sea_Yemen | Google News RSS | 109 |
| Ukraine_Black_Sea | GDELT DOC 2.1 API | 1462 |
| Ukraine_Black_Sea | Google News RSS | 103 |


In [ ]:
# Task 1 verification: head(), describe(), and missing-value checks
print("FIRMS shape:", firms.shape)
display(firms.head())
display(firms.describe(include="all"))
display(firms.isna().sum().sort_values(ascending=False).head(20))

print("News shape:", news.shape)
display(news.head())
display(news.isna().sum().sort_values(ascending=False).head(20))


## Task 2: Spatial and Temporal Analysis

Thermal detections were clustered into events using a space-time DBSCAN method. Detections were grouped if they were within approximately **10 km** and **2 days**. Each event includes centroid coordinates, start/end date, duration, total FRP, maximum brightness, number of FIRMS detections, night-detection ratio, and region.

### Thermal Event Sample

| event_id | region | centroid_latitude | centroid_longitude | start_date | end_date | duration_days | total_frp | mean_frp | max_brightness | n_detections | pct_night | conflict_associated |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| E000001 | Eastern_Mediterranean | 31.039 | 29.763 | 2024-01-01 | 2024-06-30 | 182 | 4042.480 | 2.354 | 367.000 | 1717 | 0.827 | 1 |
| E000002 | Eastern_Mediterranean | 29.904 | 40.251 | 2024-01-01 | 2024-06-17 | 169 | 10.160 | 1.693 | 350.380 | 6 | 0.667 | 1 |
| E000003 | Eastern_Mediterranean | 31.979 | 36.071 | 2024-01-01 | 2024-06-29 | 181 | 109.890 | 1.863 | 351.020 | 59 | 0.780 | 1 |
| E000004 | Eastern_Mediterranean | 36.965 | 40.362 | 2024-01-01 | 2024-06-30 | 182 | 53526.880 | 7.978 | 367.000 | 6709 | 0.591 | 1 |
| E000005 | Eastern_Mediterranean | 37.802 | 40.530 | 2024-01-01 | 2024-06-30 | 182 | 1073.370 | 6.241 | 367.000 | 172 | 0.547 | 1 |
| E000006 | Eastern_Mediterranean | 37.659 | 33.280 | 2024-01-01 | 2024-04-01 | 92 | 17.390 | 3.478 | 339.420 | 5 | 0.000 | 0 |
| E000007 | Eastern_Mediterranean | 34.919 | 40.833 | 2024-01-01 | 2024-06-30 | 182 | 513.370 | 1.497 | 356.450 | 343 | 0.901 | 1 |
| E000008 | Eastern_Mediterranean | 35.303 | 40.444 | 2024-01-01 | 2024-06-30 | 182 | 140.590 | 1.034 | 324.780 | 136 | 1.000 | 1 |

### Regional Thermal Event Summary

| region | thermal_events | mean_total_frp | median_total_frp | mean_duration_days | mean_detections |
| --- | --- | --- | --- | --- | --- |
| Ukraine_Black_Sea | 1485 | 163.822 | 9.880 | 29.867 | 26.412 |
| Persian_Gulf | 614 | 1494.059 | 13.400 | 72.212 | 279.572 |
| Eastern_Mediterranean | 469 | 251.116 | 10.860 | 48.158 | 47.539 |
| Red_Sea_Yemen | 336 | 773.205 | 8.640 | 37.354 | 95.673 |


### Task 2 Visualizations

![Monthly thermal event frequency](figures/task2_monthly_event_frequency.png)

![Mean FRP trend](figures/task2_mean_frp_trend.png)

![Day/night detections](figures/task2_daynight_detection_counts.png)

![Spatial thermal events](figures/task2_spatial_events_duration.png)

![Top hotspots](figures/task2_top_hotspots.png)


In [ ]:
# Task 2 verification tables
display(events.head())
display(events.groupby("region").agg(
    thermal_events=("event_id", "count"),
    mean_total_frp=("total_frp", "mean"),
    mean_duration_days=("duration_days", "mean")
))
display(monthly.head(12))


## Task 3: Thermal-News Correlation and Classification

Thermal events were matched to conflict news articles when the article was published within a **0-7 day temporal window**, belonged to the same region, and contained conflict-related keywords such as war, conflict, bombing, airstrike, shelling, missile, attack, troops, armed, explosion, or combat.

### Event-News Match Sample

| event_id | region | article_title | article_url | article_source | article_date | lag_days | match_reason |
| --- | --- | --- | --- | --- | --- | --- | --- |
| E000001 | Eastern_Mediterranean | Israeli army says senior Hezbollah leader killed in Lebanon airstrike | https://www.yahoo.com/news/israeli-army-says-senior-hezbollah-081347701.html | yahoo.com | 2024-05-15 09:30:00 | 135 | same region, 0-7 day window, conflict keyword |
| E000001 | Eastern_Mediterranean | Senior Hezbollah official doubts seriousness of US intention to stop war in Gaza | https://tass.com/world/1798063 | tass.com | 2024-06-04 16:30:00 | 155 | same region, 0-7 day window, conflict keyword |
| E000001 | Eastern_Mediterranean | Senior Hezbollah leader killed in Lebanon airstrike : Israel | https://www.prokerala.com/news/articles/a1531297.html | prokerala.com | 2024-05-15 08:30:00 | 135 | same region, 0-7 day window, conflict keyword |
| E000001 | Eastern_Mediterranean | Israeli Airstrike In Syria Kills Girl , Injures 10 Civilians | https://www.ndtv.com/world-news/israeli-airstrike-in-syria-kills-girl-injures-10-civilians-5775157 | ndtv.com | 2024-05-29 23:30:00 | 149 | same region, 0-7 day window, conflict keyword |
| E000001 | Eastern_Mediterranean | Four Lebanese killed in Israeli airstrike | https://www.thedailystar.net/news/world/news/four-lebanese-killed-israeli-airstrike-3603006 | thedailystar.net | 2024-05-05 19:45:00 | 125 | same region, 0-7 day window, conflict keyword |
| E000001 | Eastern_Mediterranean | Over 60 Rockets Launched At Israeli Military Bases , Says Hezbollah | https://www.ndtv.com/world-news/over-60-rockets-launched-at-israeli-military-bases-says-hezbollah-5676102 | ndtv.com | 2024-05-16 11:30:00 | 136 | same region, 0-7 day window, conflict keyword |
| E000001 | Eastern_Mediterranean | Palestinians taking shelter injured after Israeli troops shell school in Gaza | https://www.mirror.co.uk/news/world-news/palestinians-taking-shelter-injured-after-32880429 | mirror.co.uk | 2024-05-23 20:15:00 | 143 | same region, 0-7 day window, conflict keyword |
| E000001 | Eastern_Mediterranean | Car explosion kills at least one in Syria Damascus | Syria War News | https://www.aljazeera.com/news/2024/5/25/killed-in-car-explosion-in-syria-damascus | aljazeera.com | 2024-05-25 16:45:00 | 145 | same region, 0-7 day window, conflict keyword |

### Conflict Association and Coverage Summary

| region | conflict_association_rate | conflict_associated_events | total_articles | unique_sources | articles_per_thermal_event | satellite_value_score |
| --- | --- | --- | --- | --- | --- | --- |
| Ukraine_Black_Sea | 0.987 | 1466 | 1565 | 248 | 1.054 | 0.948 |
| Persian_Gulf | 0.811 | 498 | 1068 | 670 | 1.739 | 0.574 |
| Eastern_Mediterranean | 0.676 | 317 | 482 | 287 | 1.028 | 0.971 |
| Red_Sea_Yemen | 0.583 | 196 | 609 | 385 | 1.812 | 0.551 |

### Hypothesis Test

The hypothesis test compares **total FRP** between conflict-associated and non-conflict thermal events using a Welch t-test.

| test | metric | group_1 | group_1_n | group_1_mean | group_2 | group_2_n | group_2_mean | t_statistic | p_value |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Welch t-test | total_frp | conflict_associated | 2477 | 618.630 | not_conflict_associated | 427 | 13.708 | 2.314 | 0.021 |

### Machine Learning Results

| model | accuracy | precision | recall | f1 | test_rows |
| --- | --- | --- | --- | --- | --- |
| SVM | 0.917 | 0.981 | 0.921 | 0.950 | 581 |
| Decision Tree | 0.883 | 0.989 | 0.873 | 0.927 | 581 |
| Logistic Regression | 0.866 | 0.984 | 0.857 | 0.916 | 581 |
| Gaussian Naive Bayes | 0.811 | 0.995 | 0.782 | 0.876 | 581 |


### Task 3 Visualizations

![Articles by region and source](figures/task3_articles_by_region_source.png)

![Keyword heatmap](figures/task3_keyword_heatmap.png)

![SVM confusion matrix](figures/task3_confusion_matrix_svm.png)

![Logistic regression confusion matrix](figures/task3_confusion_matrix_logistic_regression.png)

![Decision tree confusion matrix](figures/task3_confusion_matrix_decision_tree.png)

![Gaussian Naive Bayes confusion matrix](figures/task3_confusion_matrix_gaussian_naive_bayes.png)


In [ ]:
# Task 3 verification tables
display(matches.head())
display(coverage.sort_values("conflict_association_rate", ascending=False))
display(hypothesis)
display(ml_results)


## Task 4: Dashboard, Insights, and Reflection

![Final dashboard](dashboard.png)

### Key Findings

The pipeline processed **265,321** cleaned FIRMS detections and clustered them into **2,904** thermal events. The strongest news-linked signal by conflict-association rate was observed in **Ukraine_Black_Sea**, with an association rate of **98.7%**. The highest average thermal intensity was observed in **Persian_Gulf**, where mean total FRP reached **1494.1 MW** per clustered event. The results support the idea that satellite thermal anomalies can act as useful early-warning signals, but they work best when interpreted alongside news and regional context.

### Shipping and Energy Implications

For a maritime shipping company, the highest-risk regions are those where thermal activity, conflict-news association, and strategic shipping or energy geography overlap. The Black Sea affects grain and energy-linked trade, the Red Sea/Yemen area affects Bab el-Mandeb and Suez-linked routing, the Persian Gulf is directly tied to oil export risk, and the Eastern Mediterranean connects conflict risk with port operations and regional energy infrastructure. A practical monitoring system should flag sudden monthly spikes in thermal events, unusually high FRP clusters, and increases in conflict-associated thermal events as triggers for route review, insurance reassessment, and fuel hedging discussions.

### Limitations and Future Work

The analysis cannot prove that a thermal anomaly was caused by conflict. FIRMS also detects natural fires, agricultural burning, industrial flares, and accidental explosions. The news-matching strategy uses region labels, keywords, and a temporal window, so it may miss articles with vague geography or match events that are only loosely related. News coverage is uneven: regions with fewer articles can look less conflict-associated even when risk is high. Future work should add ACLED or verified incident datasets, AIS vessel tracks, port disruption data, oil-price movements, better geocoding of article locations, and manual validation labels for a stronger classifier.

### Methodology Reflection

The hardest part of the pipeline was linking satellite detections to news in a defensible way. Satellite data is dense and quantitative, while news is sparse, biased, and text-based. If starting over, I would build a stronger geocoding layer, validate a sample of event-news matches by hand, and extend the observation window beyond six months to separate seasonal fire patterns from conflict-related thermal signatures more confidently.


## Final Deliverables Included

- `CE49X_Final_Project_COMPLETE_OUTPUTS.ipynb`: this complete output notebook.
- `CE49X_Final_Project_Run_All.py`: complete executable pipeline.
- `dashboard.png`: required dashboard at 300 DPI.
- `figures/`: all supporting visualizations.
- `final_data_snapshots/`: final tables and analysis summaries.
- `docker-compose.yml`: PostgreSQL Docker setup.
- `requirements.txt`: package list.
- `PRESENTATION_SCRIPT.md`: recorded presentation script.
